# Cleaning Robot - Stochastic MDP

![](images/cleaning-robot-mdp-3.png)

The following code shows the estimation of the q value function for a policy, the optimal q_star and the optimal policy for the cleaning robot problem in the strochastic case. 

In [1]:
import numpy as np


def stochastic_robot_cleaning_v1():
    # Initialization
    state = [0, 1, 2, 3, 4, 5]  # Set of states
    action = [-1, 1]  # Set of actions

    # Make all (state, action) pairs
    A, B = np.meshgrid(state, action)
    C = np.column_stack((A.flatten(), B.flatten()))
    D1 = C  # D includes all possible combinations of (state, action) pairs

    # Make all (state, action) pairs indices
    A, B = np.meshgrid(np.arange(len(state)), np.arange(len(action)))
    C = np.column_stack((A.flatten(), B.flatten()))
    D2 = C  # D includes all possible combinations of (state, action) pairs indices

    D = np.column_stack((D1, D2))
    Q = np.zeros((len(state), len(action)))  # Initial Q can be chosen arbitrarily
    Qold = Q.copy()  # Save a backup to compare later
    L = 15  # Maximum number of iterations
    gamma = 0.5  # Discounting factor
    epsilon = 0.001  # Final error to stop the algorithm

    # Deterministic Q-iteration algorithm
    for l in range(1, L + 1):
        print(f"iteration: {l}")
        for ii in range(D.shape[0]):
            XP, P, R = model(
                D[ii, 0], D[ii, 1]
            )  # Includes the next states and the probability
            I1 = np.where(D[:, 0] == XP[0])[
                0
            ]  # Find the next state in the pair-matrix D
            I2 = np.where(D[:, 0] == XP[1])[0]
            I3 = np.where(D[:, 0] == XP[2])[0]
            Q[D[ii, 2], D[ii, 3]] = (
                P[0] * (R[0] + gamma * np.max(Q[XP[0], :]))
                + P[1] * (R[1] + gamma * np.max(Q[XP[1], :]))
                + P[2] * (R[2] + gamma * np.max(Q[XP[2], :]))
            )

        if np.abs(np.sum(Q - Qold)) < epsilon:
            print("Epsilon criteria satisfied!")
            break
        else:
            Qold = Q.copy()

    # Show the results
    print("Q matrix (optimal):")
    print(Q)

    C = np.argmax(Q, axis=1)  # Finding the max values
    print("Q(optimal):")
    print(C)
    print("Optimal Policy:")
    print("*")
    print([action[C[1]], action[C[2]], action[C[3]], action[C[4]]])
    print("*")

In [2]:
# Find the possible next states together with their probability value
def model(x, u):
    if x == 0 or x == 5:
        xp = [x, x, x]
    else:
        xp = [x - u, x, x + u]
    p = [prob(x, u, xp[0]), prob(x, u, xp[1]), prob(x, u, xp[2])]
    r = [reward(x, u, xp[0]), reward(x, u, xp[1]), reward(x, u, xp[2])]
    return xp, p, r


# This function is the transition model of the robot
# The inputs are: the current state, the chosen action, and the next desired state
# The output is the probability for going to the next state considering the stochasticity
# In the deterministic case, the model gives the next state, but in the stochastic case,
# the model gives the probability for going to the next state.
def prob(x, u, xp):
    if x == 0 or x == 5:
        if xp == x:
            return 1
        else:
            return 0
    elif 1 <= x <= 4:
        if xp == x:
            return 0.15
        elif xp == x + u:
            return 0.8
        elif xp == x - u:
            return 0.05
        else:
            return 0
    else:
        return 0


# This function is the reward function for the task (stochastic)
# The inputs are: the current state, the chosen action, and the next state
# The output is the expected reward in the next state
# The reward actually doesn't depend on the chosen action, in this case
def reward(x, u, xp):
    if x != 5 and xp == 5:
        return 5
    elif x != 0 and xp == 0:
        return 1
    else:
        return 0

In [3]:
# Call the main function
stochastic_robot_cleaning_v1()

iteration: 1
iteration: 2
iteration: 3
iteration: 4
iteration: 5
iteration: 6
iteration: 7
iteration: 8
Epsilon criteria satisfied!
Q matrix (optimal):
[[0.         0.        ]
 [0.88789562 0.45747448]
 [0.4669584  0.8522684 ]
 [0.59393876 1.91539737]
 [1.34436242 4.37609178]
 [0.         0.        ]]
Q(optimal):
[0 0 1 1 1 0]
Optimal Policy:
*
[-1, 1, 1, 1]
*


![](images/q-function.png)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colormaps, patheffects
import matplotlib.cm as cm


def _draw_panel(ax, Q, states, terminals, vmin, vmax, cmap, panel_title):
    ax.set_xlim(-0.5, len(states) - 0.5)
    ax.set_ylim(-1.1, 1.3)
    ax.axis("off")
    ax.set_title(panel_title, fontsize=12)
    stroke = [patheffects.withStroke(linewidth=2.5, foreground="white")]
    denom = max(vmax - vmin, 1e-9)
    for i, s in enumerate(states):
        if s in terminals:
            ax.add_patch(mpatches.Rectangle(
                (i - 0.45, -0.5), 0.9, 1.0,
                facecolor="#d9d9d9", edgecolor="black", linewidth=0.8))
            ax.text(i, 0, "T", ha="center", va="center", fontsize=13, color="#555")
        else:
            q_left, q_right = float(Q[i, 0]), float(Q[i, 1])
            ax.add_patch(mpatches.Rectangle(
                (i - 0.45, -0.5), 0.45, 1.0,
                facecolor=cmap((q_left - vmin) / denom),
                edgecolor="black", linewidth=0.8))
            ax.add_patch(mpatches.Rectangle(
                (i, -0.5), 0.45, 1.0,
                facecolor=cmap((q_right - vmin) / denom),
                edgecolor="black", linewidth=0.8))
            ax.text(i - 0.225, 0.32, "←", ha="center", va="center",
                    fontsize=9, color="#444")
            ax.text(i + 0.225, 0.32, "→", ha="center", va="center",
                    fontsize=9, color="#444")
            ax.text(i - 0.225, -0.18, f"{q_left:.2f}", ha="center", va="center",
                    fontsize=10, color="black", path_effects=stroke)
            ax.text(i + 0.225, -0.18, f"{q_right:.2f}", ha="center", va="center",
                    fontsize=10, color="black", path_effects=stroke)
            if q_left > 0 or q_right > 0:
                if q_right >= q_left:
                    ax.annotate("", xy=(i + 0.3, 0.85), xytext=(i - 0.3, 0.85),
                                arrowprops=dict(arrowstyle="->", color="black", lw=2.5))
                else:
                    ax.annotate("", xy=(i - 0.3, 0.85), xytext=(i + 0.3, 0.85),
                                arrowprops=dict(arrowstyle="->", color="black", lw=2.5))
        ax.text(i, -0.85, f"{s}", ha="center", va="center", fontsize=11)


def plot_q_panel(Q, states, terminals, vmin, vmax, title):
    fig, ax = plt.subplots(figsize=(11, 2.6))
    cmap = colormaps["viridis"]
    _draw_panel(ax, Q, states, terminals, vmin, vmax, cmap, title)
    sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, orientation="horizontal",
                 fraction=0.06, pad=0.12, shrink=0.6, label="Q(s, a)")
    plt.tight_layout()
    plt.show()


def stochastic_robot_cleaning_traced():
    state = [0, 1, 2, 3, 4, 5]
    action = [-1, 1]
    A, B = np.meshgrid(state, action)
    C = np.column_stack((A.flatten(), B.flatten()))
    D1 = C
    A, B = np.meshgrid(np.arange(len(state)), np.arange(len(action)))
    C = np.column_stack((A.flatten(), B.flatten()))
    D2 = C
    D = np.column_stack((D1, D2))
    Q = np.zeros((len(state), len(action)))
    Qold = Q.copy()
    L = 15
    gamma = 0.5
    epsilon = 0.001
    history = [Q.copy()]
    for _ in range(1, L + 1):
        for ii in range(D.shape[0]):
            XP, P, R = model(D[ii, 0], D[ii, 1])
            Q[D[ii, 2], D[ii, 3]] = (
                P[0] * (R[0] + gamma * np.max(Q[XP[0], :]))
                + P[1] * (R[1] + gamma * np.max(Q[XP[1], :]))
                + P[2] * (R[2] + gamma * np.max(Q[XP[2], :]))
            )
        history.append(Q.copy())
        if np.abs(np.sum(Q - Qold)) < epsilon:
            break
        Qold = Q.copy()
    return history


history = stochastic_robot_cleaning_traced()
states_list = [0, 1, 2, 3, 4, 5]
terminals = {0, 5}
suptitle = "Q-value iteration (stochastic cleaning robot)"
vmax = max(1e-6, float(np.ceil(max(Q.max() for Q in history) * 10) / 10))
vmin = 0.0
print(suptitle)
for k, Q in enumerate(history):
    panel_title = "Initial (Q = 0)" if k == 0 else f"Iteration {k}"
    plot_q_panel(Q, states=states_list, terminals=terminals,
                 vmin=vmin, vmax=vmax, title=panel_title)
